# Pilot 2025 global model selection and DOE

This notebook documents a full deterministic workflow for the pilot-scale fermentation data:

1. Load the curated pilot data and the previous ethyl-acetate aroma model.
2. Diagnose state-level model adequacy before re-fitting any new structure.
3. Compare secondary-metabolite ODE variants for pyruvate, acetaldehyde, and acetate.
4. Select a final ODE structure using weighted residual metrics, information criteria, and boundary diagnostics.
5. Build the current-data Fisher Information Matrix (FIM) with Pyomo DoE-compatible scaling.
6. Rank natural-must candidate experiments using model-based DOE metrics.

The distinction used here is:

$$r(\theta)=\frac{\hat{y}(\theta)-y}{\sigma_y}$$

$$J=\frac{\partial r}{\partial \log\theta}, \qquad F=J^\top J.$$

Poor estimability means that $F$ has weak directions after a model can already reproduce the data. Poor structural adequacy means that the residuals remain biased or shape-incompatible before the FIM interpretation is meaningful.


In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

RESULTS = Path('results/global_state_model_selection_doe')
print(RESULTS.resolve())


C:\Users\ctorrealba\OneDrive - Viña Concha y Toro S.A\Documentos\Doctorado\Artículos\Artículo - Estimación_dFBA\pyomo-doe\fermentation_model\pilot_2025\results\global_state_model_selection_doe


## Data and curation

The pilot workbook is treated as natural-must data. Online CO2 files are curated as in the previous pilot workflow: unusable CO2 files for batches 25150 and 25151 are excluded, and batch 25171 is shifted so that the sustained CO2 activation point is the effective fermentation start.

Dissolved oxygen is not used as a pilot observation here because the current pilot workbook does not expose a usable DO column; O2 remains a latent state in the secondary model.


In [2]:
pd.read_csv(RESULTS / 'co2_curation_decisions.csv')

,batch,raw_rows,used_for_calibration,reason,activation_time_h,curated_rows
0,25150,56,False,excluded: CO2 file not usable for this batch,NaN,0
1,25151,56,False,excluded: CO2 file not usable for this batch,NaN,0
2,25170,22877,True,kept,0.0,22837
3,25171,20000,True,trimmed to operational CO2/process activation ...,100.0,13963


## Observation coverage

The adequacy diagnostic is only interpreted for states with actual observations. States without observations can still affect the FIM through model coupling and future DOE outputs, but their current-data residual fit is not directly testable.


In [3]:
pd.read_csv(RESULTS / 'observation_coverage.csv')

,group,state,n_observations
0,core,X,124
1,core,Xd,0
2,core,N,127
3,core,G,131
4,core,F,131
5,core,E,131
6,core,Gly,131
7,secondary,Pyr,129
8,secondary,AcAld,107
9,secondary,Acetate,0


## State-level adequacy diagnostic

For each observed state, species, and pool, the diagnostic computes:

$$RMSE=\sqrt{\frac{1}{n}\sum_i(\hat{y}_i-y_i)^2}$$

$$rRMSE=\frac{RMSE}{\max(\operatorname{median}(|y|),0.25(\max y-\min y),\epsilon)}$$

$$rBias=\frac{\operatorname{mean}(\hat{y}-y)}{\max(\operatorname{median}(|y|),0.25(\max y-\min y),\epsilon)}.$$

The flag is structural when normalized error is high and/or residuals show systematic shape incompatibility. Limited data are not interpreted as structural failure.


In [4]:
initial = pd.read_csv(RESULTS / 'adequacy_initial_current.csv')
cols = [c for c in ['group','state','species','pool','n','relative_rmse','relative_bias','corr','adequacy_flag','adequacy_reason'] if c in initial.columns]
initial.sort_values(['adequacy_flag','relative_rmse'], ascending=[False, False])[cols]


,group,state,species,pool,n,relative_rmse,relative_bias,corr,adequacy_flag,adequacy_reason
4,core,N,NaN,NaN,127,0.938200,-0.292536,0.701183,weak_fit,high normalized error but limited systematic bias
11,aroma,ethyl_octanoate:condensate,ethyl_octanoate,condensate,44,0.737966,-0.267528,0.649098,weak_fit,high normalized error but limited systematic bias
8,aroma,ethyl_acetate:condensate,ethyl_acetate,condensate,17,4.860131,4.460879,0.477578,structure_warning,normalized RMSE exceeds the observation scale
17,co2,CO2_rate_shape,NaN,NaN,191,1.242877,-0.563244,0.328081,structure_warning,CO2 signal shape is poorly correlated with eth...
14,aroma,isoamyl_acetate:condensate,isoamyl_acetate,condensate,44,0.811653,-0.371141,0.445504,structure_warning,large normalized error with systematic bias
9,aroma,ethyl_acetate:retained,ethyl_acetate,retained,17,0.575972,-0.437476,0.019582,structure_warning,poor shape correlation and high normalized error
12,aroma,ethyl_octanoate:retained,ethyl_octanoate,retained,44,0.648791,-0.178025,0.343862,adequate,error is within the pragmatic adequacy thresholds
13,aroma,ethyl_octanoate:total,ethyl_octanoate,total,52,0.564104,-0.184798,0.639452,adequate,error is within the pragmatic adequacy thresholds
1,core,F,NaN,NaN,131,0.544920,-0.237967,0.877365,adequate,error is within the pragmatic adequacy thresholds
2,core,G,NaN,NaN,131,0.538618,0.039191,0.906203,adequate,error is within the pragmatic adequacy thresholds


## Secondary-state structural variants

The selected aroma model from the previous workflow is kept fixed while secondary-metabolite structure is tested. This avoids confusing ethyl-acetate structural error with pyruvate, acetaldehyde, or acetate structural error.

The common gating terms are:

$$\phi_N=\frac{N}{N+K_N}, \qquad \phi_{stat}=1-\phi_N,$$

$$g_{O2}=\frac{O_2}{O_2+K_{O2}}, \qquad \phi_{ana}=\frac{K_{O2}}{O_2+K_{O2}}, \qquad \phi_E=\frac{E}{E+K_E}.$$

Candidate structures are compared using robust log-parameter least squares. The model-selection score is:

$$Score = BIC + P_{bounds}+P_{adequacy}.$$

This penalizes models that fit by driving parameters to bounds or by keeping large state-level residual bias.


In [5]:
pd.read_csv(RESULTS / 'secondary_model_selection_summary.csv')

,model,data_wsse,n_data_residuals,n_parameters,active_bound_count,aicc,bic,state_penalty,selection_score,description,selected
0,secondary_full_chem_o2fixed,1816.799919,236,12,1,1842.199022,1882.365901,25.0,1907.365901,Full chemical secondary layer with O2 transfer...,True
1,secondary_redox_o2,1882.012763,236,9,2,1900.809223,1931.187249,50.0,1981.187249,Adds a pyruvate-to-acetaldehyde source and an ...,False
2,secondary_phase_split,2587.544326,236,9,1,2606.340786,2636.718812,25.0,2661.718812,Adds stationary-phase production terms for pyr...,False
3,secondary_reduced_o2fixed,2798.517423,236,7,1,2813.008651,2836.764245,25.0,2861.764245,Current reduced v2 structure: pyruvate and ace...,False
4,secondary_acetate_assimilation,2798.523467,236,8,1,2815.157828,2842.234121,25.0,2867.234121,Adds acetate assimilation during nitrogen-asso...,False


## Selected secondary structure

Selected structure: `secondary_full_chem_o2fixed`

Full chemical secondary layer with O2 transfer parameters fixed. This is the upper-complexity check: it is accepted only if its improved residuals justify the extra degrees of freedom.

$$\frac{dPyr}{dt}=(k_{PyrS,N}\phi_N+k_{PyrS,stat}\phi_{stat})q_S+k_{PyrO2}g_{O2}X-k_{PyrDrain}Pyr X(0.25+\phi_{stat}+0.5\phi_E)$$

$$\frac{dAcAld}{dt}=k_{AldPyr}PyrX+(k_{AldS,N}\phi_N+k_{AldS,stat}\phi_{stat})q_S+k_{AldO2}g_{O2}X-k_{AldRed}AcAldX(\phi_{ana}+0.25\phi_{stat})-k_{AcAld}AcAldXg_{O2}$$

$$\frac{dAcetate}{dt}=\frac{k_{AcAld}AcAldXg_{O2}}{1000}+k_{AcStress}X\phi_E-k_{AcAssim}AcetateX\phi_N$$


In [6]:
pd.read_csv(RESULTS / 'theta_selected_global_model.csv', index_col=0).head(60)

,0
mu0,0.068971
sN,8.751446
qN,0.015772
qXG,0.081491
qXF,0.070933
betaG0,1.186336
sG,0.097455
betaF0,0.314510
sF,0.178295
qEG,1.112514


## Final adequacy after structural selection

The table below repeats the adequacy diagnostic after substituting the selected secondary structure into the global model. States that remain flagged are not automatically discarded; instead, they indicate where either additional experimental excitation, additional measurements, or stronger literature priors are needed before those parameters should be used as flexible MPCC degrees of freedom.


In [7]:
final = pd.read_csv(RESULTS / 'adequacy_final_selected.csv')
cols = [c for c in ['group','state','species','pool','n','relative_rmse','relative_bias','corr','adequacy_flag','adequacy_reason'] if c in final.columns]
final.sort_values(['adequacy_flag','relative_rmse'], ascending=[False, False])[cols]


,group,state,species,pool,n,relative_rmse,relative_bias,corr,adequacy_flag,adequacy_reason
4,core,N,NaN,NaN,127,0.938200,-0.292536,0.701183,weak_fit,high normalized error but limited systematic bias
11,aroma,ethyl_octanoate:condensate,ethyl_octanoate,condensate,44,0.737966,-0.267528,0.649098,weak_fit,high normalized error but limited systematic bias
8,aroma,ethyl_acetate:condensate,ethyl_acetate,condensate,17,4.860131,4.460879,0.477578,structure_warning,normalized RMSE exceeds the observation scale
17,co2,CO2_rate_shape,NaN,NaN,191,1.242877,-0.563244,0.328081,structure_warning,CO2 signal shape is poorly correlated with eth...
14,aroma,isoamyl_acetate:condensate,isoamyl_acetate,condensate,44,0.811653,-0.371141,0.445504,structure_warning,large normalized error with systematic bias
9,aroma,ethyl_acetate:retained,ethyl_acetate,retained,17,0.575972,-0.437476,0.019582,structure_warning,poor shape correlation and high normalized error
12,aroma,ethyl_octanoate:retained,ethyl_octanoate,retained,44,0.648791,-0.178025,0.343862,adequate,error is within the pragmatic adequacy thresholds
13,aroma,ethyl_octanoate:total,ethyl_octanoate,total,52,0.564104,-0.184798,0.639452,adequate,error is within the pragmatic adequacy thresholds
1,core,F,NaN,NaN,131,0.544920,-0.237967,0.877365,adequate,error is within the pragmatic adequacy thresholds
2,core,G,NaN,NaN,131,0.538618,0.039191,0.906203,adequate,error is within the pragmatic adequacy thresholds


## FIM and estimability

The FIM is built from current-data residuals using finite differences in log-parameter space:

$$J_{ij}\approx\frac{r_i(\theta_j e^h)-r_i(\theta_j e^{-h})}{2h}.$$

The eigenspectrum of $F$ diagnoses practical identifiability. The weakest eigenvectors identify confounded parameter combinations. The approximate log-standard deviation is read from a stabilized inverse FIM:

$$\Sigma_{\log\theta}\approx F^{-1}.$$

Parameters are classified as well estimated, moderate, weak but actionable, or weak/confounded using the same thresholds as the previous pilot aroma workflow.


In [8]:
json.load(open(RESULTS / 'target_parameters_global_selected.json'))

['mu0',
 'qN',
 'betaG0',
 'betaF0',
 'qEG',
 'qEF',
 'iG',
 'iE',
 'Kd0',
 'gammaG0',
 'gammaF0',
 'kPyrS_N',
 'kPyrS_stat',
 'kPyrO2',
 'kPyrDrain',
 'kAldPyr',
 'kAldS_N',
 'kAldS_stat',
 'kAldO2',
 'kAldRed',
 'kAcAld',
 'kAcStress',
 'kAcAssim',
 'k_EA_growth',
 'k_EA_stationary',
 'k_IAA_growth',
 'k_IAA_stationary',
 'k_EO_growth',
 'k_EO_stationary',
 'alpha_EA_loss',
 'alpha_IAA_loss',
 'alpha_EO_loss',
 'k_EA_XE',
 'k_EA_XE_Nlim']

In [9]:
pd.read_csv(RESULTS / 'parameter_estimability_global_current.csv')

,analysis,parameter,theta,std_log_approx,approx_95_multiplier,fim_diag,active_bound,classification
0,global_current,mu0,0.068971,0.011790,1.023378e+00,35107.659799,False,well_estimated
1,global_current,qN,0.015772,0.015248,1.030336e+00,20515.428764,False,well_estimated
2,global_current,betaG0,1.186336,0.042704,1.087302e+00,51222.548290,False,well_estimated
3,global_current,betaF0,0.314510,0.212689,1.517204e+00,3954.064220,False,well_estimated
4,global_current,qEG,1.112514,0.040156,1.081886e+00,50444.853660,False,well_estimated
5,global_current,qEF,1.278010,0.101136,1.219238e+00,16883.673967,False,well_estimated
6,global_current,iG,0.013971,0.208470,1.504711e+00,1449.759125,False,well_estimated
7,global_current,iE,0.015660,0.087190,1.186364e+00,6313.577314,False,well_estimated
8,global_current,Kd0,0.000656,0.082754,1.176094e+00,210.353157,False,well_estimated
9,global_current,gammaG0,0.070310,0.093788,1.201805e+00,8712.441757,False,well_estimated


In [10]:
pd.read_csv(RESULTS / 'weak_directions_global_current.csv')

,analysis,weak_direction,eigenvalue,dominant_parameters,dominant_abs_loadings
0,global_current,1,-2.792552e-14,"kAcStress, kAcAssim, k_EA_XE, k_EA_growth, kAl...","0.993, 0.121, 0.000, 0.000, 0.000, 0.000, 0.00..."
1,global_current,2,-7.339981e-15,"kAcAssim, kAcStress, k_EA_XE, k_EA_growth, kAl...","0.993, 0.121, 0.000, 0.000, 0.000, 0.000, 0.00..."
2,global_current,3,1.883117e-08,"k_EA_XE, k_EA_growth, k_EA_XE_Nlim, kAldRed, k...","0.998, 0.057, 0.000, 0.000, 0.000, 0.000, 0.00..."
3,global_current,4,1.220398e-04,"k_EA_growth, k_EA_XE, k_EA_stationary, kAldRed...","0.998, 0.057, 0.003, 0.001, 0.000, 0.000, 0.00..."
4,global_current,5,8.468546e-04,"kAldRed, kAldPyr, kPyrS_stat, kAldO2, kAldS_st...","1.000, 0.012, 0.006, 0.005, 0.003, 0.002, 0.00..."
5,global_current,6,3.628690e-02,"kPyrS_stat, kPyrO2, kPyrDrain, kAldS_stat, kAl...","0.997, 0.070, 0.009, 0.008, 0.006, 0.006, 0.00..."
6,global_current,7,2.476057e-01,"gammaF0, gammaG0, betaF0, iG, qEF, iE, qEG, k_...","0.999, 0.041, 0.022, 0.015, 0.009, 0.008, 0.00..."
7,global_current,8,6.839166e+00,"k_EA_stationary, k_EA_XE_Nlim, alpha_EA_loss, ...","0.989, 0.122, 0.076, 0.020, 0.011, 0.007, 0.00..."


## Model-based DOE

Candidate natural-must designs are ranked by adding each candidate FIM to the current-data FIM. The hybrid score keeps D-optimality as the main objective while penalizing weak minimum eigen-directions:

$$\Phi_{hybrid}=\log\det(F)-2|\log(\lambda_{min}/\lambda_{max})|-0.05\log(\operatorname{tr}(F^{-1})).$$

The greedy campaign then adds the experiment that maximizes this score at each step. This is homologous to the previous `doe_multiexperiment` logic: current-data FIM acts as the prior information matrix and candidate experiments add information sequentially.


In [11]:
pd.read_csv(RESULTS / 'candidate_ranking_global_selected.csv').head(15)

,candidate,family,medium,horizon_h,temperature_segments,N_pulses_kg_m3,candidate_logdet,candidate_min_eigenvalue,candidate_max_eigenvalue,candidate_min_relative_eigenvalue,...,var_ratio_k_EA_XE_Nlim,var_reduction_k_EA_XE_Nlim,target_mean_var_reduction,target_worst_var_reduction,aroma_mean_var_reduction,aroma_worst_var_reduction,hybrid_score,dopt_score,eopt_score,rationale
0,natural_pilot_cold_to_warm_earlyN,temperature_N,natural,300.0,"14, 16, 22, 20",30h:0.045,49.745403,7.892330e-09,16990.563332,4.645125e-13,...,0.833489,0.166511,0.286107,0.081641,0.244485,0.096308,72.971636,132.095649,-17.590180,Cold start followed by warm transition and ear...
1,natural_pilot_EA_cold_warm_Nsplit,EA_temperature_Nsplit,natural,300.0,"13, 18, 23, 19",32h:0.025; 80h:0.035,49.261221,3.966454e-10,16700.150644,2.375101e-14,...,0.847068,0.152932,0.287802,0.084217,0.248990,0.109114,72.405610,131.716465,-17.675041,Cold start then warm acceleration with split n...
2,natural_pilot_EA_cold_retention_noN,EA_retention_reference,natural,300.0,"13, 13, 16, 18",NaN,48.775290,2.553132e-09,22164.275994,1.151913e-13,...,0.804906,0.195094,0.335215,0.078422,0.264602,0.112203,72.202479,132.481642,-18.139846,Low-temperature no-pulse comparator to decoupl...
3,natural_pilot_low_temp_aroma_retention,aroma_retention,natural,300.0,"13, 15, 16, 16",54h:0.035,47.534414,2.459512e-09,18662.807672,1.317868e-13,...,0.850734,0.149266,0.298901,0.083601,0.243621,0.104897,71.492626,130.891842,-17.727696,Cold profile to contrast aroma retention again...
4,natural_pilot_midN_temperature_step,temperature_N,natural,300.0,"16, 20, 23, 18",54h:0.04,47.102443,3.069950e-09,15871.772981,1.934220e-13,...,0.809849,0.190151,0.267409,0.037069,0.256792,0.090997,71.447300,131.793038,-18.193537,Temperature step with mid-growth nitrogen pert...
5,natural_pilot_noN_dynamic_temperature,temperature,natural,300.0,"15, 22, 18, 22",NaN,44.842388,3.083631e-09,15049.579480,2.048982e-13,...,0.825077,0.174923,0.286768,0.060075,0.271382,0.111317,70.716062,131.698524,-18.528192,Temperature-only perturbation for settings whe...
6,natural_pilot_two_step_N_ladder,N_timing,natural,300.0,"17, 19, 21, 18",30h:0.03; 78h:0.035,44.104974,2.725422e-09,16448.881849,1.656904e-13,...,0.808959,0.191041,0.251969,0.029669,0.253124,0.091006,70.639833,131.312592,-18.383908,Two smaller nitrogen pulses to separate early ...
7,natural_pilot_high_rate_strip,co2_aroma,natural,300.0,"21, 24, 23, 19",30h:0.03,39.953516,2.456347e-11,16054.364711,1.530018e-15,...,0.808475,0.191525,0.307184,0.060554,0.263917,0.095175,70.188112,132.181459,-19.008089,High-rate natural fermentation to excite CO2 s...
8,natural_pilot_lateN_stationary_probe,N_timing,natural,300.0,"18, 20, 20, 18",80h:0.05,41.938463,6.628750e-10,15816.787061,4.190959e-14,...,0.839159,0.160841,0.254034,0.039952,0.249401,0.093947,69.332003,130.370930,-18.576023,Late nitrogen addition to test stationary/grow...
9,natural_pilot_EA_warm_early_noN,EA_temperature_Nstress,natural,300.0,"24, 23, 19, 17",NaN,37.745936,2.037204e-11,15438.237174,1.319583e-15,...,0.811890,0.188110,0.347356,0.046911,0.275291,0.069679,69.150384,132.581293,-19.759204,Warm early natural fermentation without nutrie...


In [12]:
pd.read_csv(RESULTS / 'selected_campaign_hybrid_global_selected.csv')

,objective,campaign_order,candidate,family,medium,horizon_h,temperature_segments,N_pulses_kg_m3,score,rationale,...,var_ratio_alpha_EO_loss,var_reduction_alpha_EO_loss,var_ratio_k_EA_XE,var_reduction_k_EA_XE,var_ratio_k_EA_XE_Nlim,var_reduction_k_EA_XE_Nlim,target_mean_var_reduction,target_worst_var_reduction,aroma_mean_var_reduction,aroma_worst_var_reduction
0,hybrid,1,natural_pilot_cold_to_warm_earlyN,temperature_N,natural,300.0,"14, 16, 22, 20",30h:0.045,72.971636,Cold start followed by warm transition and ear...,...,0.743531,0.256469,0.810850,0.189150,0.833489,0.166511,0.286107,0.081641,0.244485,0.096308
1,hybrid,2,natural_pilot_EA_warm_early_noN,EA_temperature_Nstress,natural,300.0,"24, 23, 19, 17",NaN,82.941966,Warm early natural fermentation without nutrie...,...,0.547958,0.452042,0.704646,0.295354,0.696384,0.303616,0.477910,0.123989,0.396150,0.219528
2,hybrid,3,natural_pilot_EA_cold_retention_noN,EA_retention_reference,natural,300.0,"13, 13, 16, 18",NaN,89.429391,Low-temperature no-pulse comparator to decoupl...,...,0.502564,0.497436,0.594264,0.405736,0.600912,0.399088,0.564692,0.175032,0.477004,0.279056
3,hybrid,4,natural_pilot_EA_cold_warm_Nsplit,EA_temperature_Nsplit,natural,300.0,"13, 18, 23, 19",32h:0.025; 80h:0.035,94.500098,Cold start then warm acceleration with split n...,...,0.443280,0.556720,0.529387,0.470613,0.548548,0.451452,0.607446,0.203735,0.529215,0.333790
4,hybrid,5,natural_pilot_warm_to_cool_noN,temperature,natural,300.0,"22, 22, 17, 16",NaN,98.375781,"Warm early phase to excite growth and CO2, the...",...,0.392495,0.607505,0.483708,0.516292,0.500431,0.499569,0.645221,0.220701,0.571771,0.357697
5,hybrid,6,natural_pilot_low_temp_aroma_retention,aroma_retention,natural,300.0,"13, 15, 16, 16",54h:0.035,101.580236,Cold profile to contrast aroma retention again...,...,0.363805,0.636195,0.440745,0.559255,0.466330,0.533670,0.672595,0.241772,0.602272,0.394688


## Interpretation

The final model combines:

- The primary fermentation and glycerol model used in the previous pilot calibration.
- The selected secondary structure `secondary_full_chem_o2fixed`.
- The selected aroma structure `ea_ethanol_nlimited` with Antoine plus UNIFAC water-ethanol partition and CO2-driven stripping.

The practical decision rule is:

- If a state is structurally adequate and a parameter is weak in the FIM, prioritize DOE excitation.
- If a state remains structurally flagged, do not interpret weak FIM directions as only a sampling/design problem; first improve model structure, measurement interpretation, or priors.
- If a parameter is active at a bound and dominates weak directions, fix or regularize it before using the model as an MPCC constraint.
